# Structured Output
This notebook primarily shows examples of structured output capabilities comparing to unstructured responses

In [2]:
from langchain_groq.chat_models import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
from pydantic import BaseModel, Field, SecretStr
import os

In [3]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
if api_key:
    GROQ_API_KEY = SecretStr(api_key)

In [4]:
base_llm = ChatGroq(model="llama-3.1-8b-instant", api_key=GROQ_API_KEY)

In [5]:
llm_prompt = ChatPromptTemplate.from_template("""
    You are an intelligent AI assistant designed to analyze user information. Given an unstructured info of a user you 
    are to provide key features about the user.
    user info: {user}
""")

In [6]:
base_chain = llm_prompt | base_llm

In [7]:
input_query = """
My name is Ilia, I'm 21 years old. I do live in Minsk and work as ML/DS engineer.
"""

In [9]:
base_llm_response = base_chain.invoke({
    "user": input_query
})
base_llm_response.content

"Based on the provided user information, I've analyzed key features about the user:\n\n1. **Name**: Ilia\n2. **Age**: 21 years old\n3. **Location**: Minsk\n4. **Occupation**: ML/DS (Machine Learning/Data Science) Engineer\n\nAdditionally, I can make some inferences about the user based on their occupation:\n\n1. **Field of Study**: Ilia likely has a strong background in computer science, mathematics, or a related field, given their profession as a ML/DS engineer.\n2. **Technical Skills**: Ilia is likely proficient in programming languages such as Python, R, or Julia, and has experience with machine learning libraries and tools like TensorFlow, PyTorch, or scikit-learn.\n3. **Professional Experience**: As a 21-year-old ML/DS engineer, Ilia is likely in their early stages of their career, with potentially 1-3 years of experience in the field.\n\nPlease note that these inferences are based solely on the provided information and may not be entirely accurate."

Basically the response is not that bad, I even like it. But there's a huge problem with it.

Imagine we are building a system where we have to automatically upload user info to a relational database based on their short description via LLM.

As the response reveals it is clearly seen that this task becomes unreachable, however we still can implement something like key-woard search or Regex filtration, but LLM's response is not deterministic and it can change it's content, can change keys we would likely search by and so on.

But what if we had some algorithm that would force LLM to follow specific structure. This approach is called structured output.


In [13]:
class UserInfo(BaseModel):
    name: str = Field(default="", description="User name")
    age: int = Field(default=1, ge=1, description="User age")
    location: str = Field(default="", description="User location")
    job:str = Field(default="", description="User occupation")

This way via `Pydantic` library we specified an output schema that the LLM will have to follow.

The question is why Pydantic instead of, say, `dataclass` or `TypedDict`. Well the reason is that neither `dataclass` nor `TypedDict` enforce the fields to strictly follow datatypes provided, whereas `Pydantic` takes a validation step before assigning values to fields.

For instance we could have specified the same schema with `dataclass` like this.
```python
@dataclass
Class UserInfo(BaseModel):
    name: str
    age: int
    location: str
    job: str
```

The thing is if we tried to assign a string to `age`, Python would not throw an error.

So we could easily do like:
```python
user_info = UserInfo(
    name="Ilia",
    age="21",
    location="Minsk",
    job="ML/DS Engineer"
)
```
and nothing bad would happen.

So the backend fails because it expects age to be `int`, not `str`

In [14]:
structured_chain = llm_prompt | base_llm.with_structured_output(UserInfo)

One thing to notice is that `with_structured_output` expects one optional argument: `method` with 3 possible values:
- **function_calling**
- **json_schema**
- **json_mode**

**1. function_calling**

This is as if model was forced to use tool to generate structured output. So that schema eventually becomes a **tool**. It looks like this:
```python
@tool
def get_user_info(name:str, age:int, location:str, job:str):
    unvalidated_schema = {
        "name": name,
        "age": age,
        "location": location,
        "job": job}

    user = UserInfo().model_validate_json(unvalidated_schema)
    return user
``` 

**2. json_mode**

This forces the model to generate valid json syntax with opening and closing braces but does not guarantee valid fields and values.

The thing is that by using this method you MUST specify schema in prompt. You would do like this:

```python
prompt = """You are...
prompt here

Generate a response according to JSON schema with following keys:
**name**: a user name
**age**: a user age
**location**: user location (where they live)
**job**: user job
"""
```
In such a way the model will only try to follow the schema.

**3. json_schema**

This a strict and a little bit complicated and tricky approach.

Instead of validating after generation it validates model ouputs meanwhile via a CFG (Context Free Grammar). About CFG [here](https://www.geeksforgeeks.org/theory-of-computation/what-is-context-free-grammar/)

What it actually does is creates a set of rules due to which some set of tokens are considered illegal and can never be generated.

When generating tokens the model calculates probabilities and sets probability of a token to 0 whenever it's considered 'illegal'.

Here's how set of rules could look like for `UserInfo`:
```
root        ::= "{" space members space "}"
members     ::= pair | pair "," members
pair        ::= key ":" value

key         ::= "\"name\"" | "\"age\"" | "\"location\"" | "\"job\""

value       ::= string_val | int_val
string_val  ::= "\"" chars "\""
int_val     ::= [1-9] [0-9]*
```

Breakdown:
1. First steep is root rules. It says that the response will start with "{" and end with "}". Thus the mechanism will mask all the tokens in vocab except for "{" and it will gain a probability of 1. The same will happen when generating the very last token - "}"
2. `root` rules says that inbetween there should be `space` `members` `space`.
    - `space` is simply a whitespace
    - `members` is a lower level part of hierarchy
3. `members` rules say that each member consists of `pair` or `pair "," members` where the first is for the last field in json object
4. `pair` rules say that each pair consists of `key ":" value`. It's a typical format for us to see keys and values in dictionaries.
5. `key` rules say that `key` can be one of `["name", "age", "location", "job"]`
6. `value` rules say that `value` can only be a string or an integer.
7. For `string_val` it psecifies that it consists of any possible chars.
8. For `int_val` it says that number should have no leading zeros.
In such a way the model itself does not produce pure json object, it generates a json-like string which resembles json and then is validated by `UserInfo`

In [15]:
structured_response = structured_chain.invoke({
    "user":input_query
})
structured_response

UserInfo(name='Ilia', age=21, location='Minsk', job='ML/DS engineer')

In [17]:
structured_response.model_dump_json()

'{"name":"Ilia","age":21,"location":"Minsk","job":"ML/DS engineer"}'

The beautiful thing is that we can now dump it into json and give to backend for processing